In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sys 

import analysis_utils
sys.path.append("../")

import checkpoint
import generate_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
unstructured = checkpoint.CheckPoint(path="../models/MLC_batchunstruct_dallstudy1_nep20.pt", device=DEVICE)
unstructured_query_first = checkpoint.CheckPoint(path="../models/MLC_batchunstruct_qf_dallstudy1_nep20.pt", device=DEVICE)
by_trans = checkpoint.CheckPoint(path="../models/MLC_batchbytrans_dallstudy1_nep20.pt", device=DEVICE)
by_alph = checkpoint.CheckPoint(path="../models/MLC_batchbyalph_dallstudy1_nep20.pt", device=DEVICE)
by_both = checkpoint.CheckPoint(path="../models/MLC_batchbyboth_dallstudy1_nep20.pt", device=DEVICE)
all_training_runs = [unstructured, unstructured_query_first, by_trans, by_alph, by_both]

In [31]:
data = [train_run.args for train_run in all_training_runs]
for i, train_run in enumerate(all_training_runs):
    data[i]["loss"] = round(train_run.checkpoint["best_val_loss"], 3)
    data[i]["accuracy in-dist"] = round(np.max([x["in"] for x in train_run.checkpoint["val_accuracy"]]), 3)
    data[i]["accuracy out-of-dist"] = round(np.max([x["out-of"] for x in train_run.checkpoint["val_accuracy"]]), 3)
df = pd.DataFrame(data)
print("Shared Arguments:")
shared_arguments = df.loc[:, df.nunique() == 1].drop(["dir_model", "save_best", "save_best_skip", "resume", "print_batches"], axis=1).rename(columns={"dir_data": "dataset"})
shared_arguments.loc[0,:].to_markdown("table_batching_shared_params.md")
shared_arguments.loc[0,:]

Shared Arguments:


dataset            data/all_transformations_study1
batch_size                                      32
nepochs                                         20
lr                                           0.001
lr_end_factor                                 0.05
lr_warmup                                     True
nlayers_encoder                                  3
nlayers_decoder                                  3
nheads                                           8
emb_size                                       128
ff_mult                                          4
dropout                                        0.1
act                                           gelu
Name: 0, dtype: object

In [17]:
df.loc[:, df.nunique() > 1]

,filename_model,sampling_method,query_first,loss,accuracy in-dist,accuracy out-of-dist
0,MLC_batchunstruct_dallstudy1_nep20.pt,unstructured,False,1.079,0.680,0.118
1,MLC_batchunstruct_qf_dallstudy1_nep20.pt,unstructured,True,1.090,0.623,0.086
2,MLC_batchbytrans_dallstudy1_nep20.pt,transformation,False,1.203,0.640,0.121
3,MLC_batchbyalph_dallstudy1_nep20.pt,alphabet,False,1.112,0.618,0.084
4,MLC_batchbyboth_dallstudy1_nep20.pt,both,False,1.257,0.602,0.098


In [18]:
df.loc[:, df.nunique() > 1].to_markdown("table_batching.md", index=False)